# add-cxr-to-multimodal-task
---
## Setup

In [1]:
%%capture

from datetime import datetime
from typing import Any, Dict, List, Optional
import os
import polars as pl

# PyHealth Packages
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks.multimodal_mimic4 import ClinicalNotesMIMIC4, ClinicalNotesICDLabsMIMIC4, ClinicalNotesICDLabsCXRMIMIC4
from pyhealth.tasks.base_task import BaseTask

# Load MIMIC4 Files
# There's probably better ways dealing with this on the cluster, but working locally for now 
# (see: https://github.com/sunlabuiuc/PyHealth/blob/master/examples/mortality_prediction/multimodal_mimic4_minimal.py)

TASK = "ClinicalNotesICDLabsCXRMIMIC4" # The idea here is that we want additive tasks so we can evaluate the value in adding more modalities

PYHEALTH_REPO_ROOT = '/Users/wpang/Desktop/PyHealth'

EHR_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/physionet.org/files/mimiciv/2.2")
NOTE_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/physionet.org/files/mimic-iv-note/2.2")
CXR_ROOT = os.path.join(PYHEALTH_REPO_ROOT,"local_data/local/data/physionet.org/files/mimic-cxr-jpg/2.0.0")
CACHE_DIR = os.path.join(PYHEALTH_REPO_ROOT,"local_data/local/data/wp/pyhealth_cache")

DEV_MODE = True

if __name__ == "__main__":

    if TASK == "ClinicalNotesMIMIC4": # A bit janky setup at the moment and open to iteration, but conveys the point for now
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )
    
    elif TASK == 'ClinicalNotesICDLabsMIMIC4':
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )

    elif TASK == 'ClinicalNotesICDLabsCXRMIMIC4':
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                cxr_root=CXR_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cxr_tables=["metadata", "negbio"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )

In [2]:
# negbio_patient_ids = (                                                                                                
#       dataset.global_event_df                                                                                           
#       .filter(pl.col("event_type") == "negbio")                                                                         
#       .select("patient_id")                                                                                             
#       .unique()                                                                                                         
#       .collect(engine="streaming")
#       .to_series()                                                                                                      
#       .to_list()                                                                                                        
#   )  

# OR 
# dataset.unique_patient_ids[2]

ID = "15092875"
print(f"Patient ID is {ID}")

Patient ID is 15092875


In [3]:
# Single patient
patient = dataset.get_patient(ID)  

### XRay

In [4]:
negbio_events = patient.get_events(event_type="negbio")
print(negbio_events)

[Event(event_type='negbio', timestamp=datetime.datetime(2140, 9, 19, 19, 54, 24), attr_dict={'dicom_id': '1f99a5c7-4b8b443a-3f65f9ac-be43fafa-9034393c', 'study_id': '57299949', 'atelectasis': None, 'cardiomegaly': None, 'consolidation': None, 'edema': None, 'enlarged cardiomediastinum': None, 'fracture': None, 'lung lesion': None, 'lung opacity': None, 'no finding': None, 'pleural effusion': None, 'pleural other': None, 'pneumonia': '0.0', 'pneumothorax': None, 'support devices': None}), Event(event_type='negbio', timestamp=datetime.datetime(2140, 9, 19, 19, 54, 24), attr_dict={'dicom_id': 'c8844d09-c90ad8a1-bdb44323-75d062c0-722cf854', 'study_id': '57299949', 'atelectasis': None, 'cardiomegaly': None, 'consolidation': None, 'edema': None, 'enlarged cardiomediastinum': None, 'fracture': None, 'lung lesion': None, 'lung opacity': None, 'no finding': None, 'pleural effusion': None, 'pleural other': None, 'pneumonia': '0.0', 'pneumothorax': None, 'support devices': None}), Event(event_typ

In [5]:
metadata_events = patient.get_events(event_type="metadata")
print(metadata_events)

[Event(event_type='metadata', timestamp=datetime.datetime(2140, 9, 19, 19, 54, 24), attr_dict={'image_path': '/Users/wpang/Desktop/PyHealth/local_data/local/data/physionet.org/files/mimic-cxr-jpg/2.0.0/files/p15/p15092875/s57299949/1f99a5c7-4b8b443a-3f65f9ac-be43fafa-9034393c.jpg', 'dicom_id': '1f99a5c7-4b8b443a-3f65f9ac-be43fafa-9034393c', 'study_id': '57299949', 'performedprocedurestepdescription': 'CHEST (PA AND LAT)', 'viewposition': 'AP', 'rows': '2544', 'columns': '3056', 'procedurecodesequence_codemeaning': 'CHEST (PA AND LAT)', 'viewcodesequence_codemeaning': 'antero-posterior', 'patientorientationcodesequence_codemeaning': 'Erect'}), Event(event_type='metadata', timestamp=datetime.datetime(2140, 9, 19, 19, 54, 24), attr_dict={'image_path': '/Users/wpang/Desktop/PyHealth/local_data/local/data/physionet.org/files/mimic-cxr-jpg/2.0.0/files/p15/p15092875/s57299949/c8844d09-c90ad8a1-bdb44323-75d062c0-722cf854.jpg', 'dicom_id': 'c8844d09-c90ad8a1-bdb44323-75d062c0-722cf854', 'study_